# Epidemiology Data Web Scraper

**EXACT MATCH TO WEB PLATFORM** - Same deep-dive extraction logic.

Simple web scraping notebook to extract epidemiology datapoints (incidence, prevalence, counts, percentages) from public sources.

**Features (matching web platform):**
- **6-strategy deep-dive**: Current page → Search results (8 links) → Articles (8 links) → References (5 links) → Company pages (8 links) → Fallback links
- **Persistent extraction**: Keeps trying until datapoints found (max_depth=5, retries with lower threshold)
- **Recursive following**: For search/journal pages, does BOTH immediate extraction AND recursive deep-dive
- **Extracts**: Numbers, percentages, rates from 35 tables, 120k chars, 250 tags
- **Supports**: Google Scholar, journal sites, company websites, references

**Usage:**
1. Set your `indication` and `country` below
2. Add source URLs to `SOURCE_URLS` list
3. Run all cells
4. Results saved to CSV

**Note**: This notebook uses the EXACT SAME extraction logic as the web platform's `link_value_extractor.py`

In [ ]:
# Install required packages (run once)
# !pip install requests beautifulsoup4 pandas lxml

In [ ]:
import re
import time
import pandas as pd
from typing import List, Optional, Tuple, Set
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# Configuration
INDICATION = "CLL (Chronic Lymphocytic Leukemia)"  # Change this
COUNTRY = "US"  # Change this

# Source URLs to scrape
SOURCE_URLS = [
    "https://scholar.google.com/scholar?q=CLL+Chronic+Lymphocytic+Leukemia+epidemiology+US",
    "https://www.thelancet.com/action/doSearch?AllField=CLL+epidemiology",
    "https://www.nature.com/search?q=CLL+epidemiology",
    "https://www.cancer.gov/about-cancer/understanding/statistics",
    "https://seer.cancer.gov/statistics-network/explorer",
    # Add more URLs here
]

REQUEST_DELAY = 1.0  # Delay between requests (seconds)
REQUEST_TIMEOUT = 20
USER_AGENT = "Mozilla/5.0 (compatible; EpidemiologyScraper/1.0)"

In [ ]:
# Patterns and keywords - MATCHING WEB PLATFORM
SEARCH_HOSTS = ("scholar.google.com", "google.com", "bing.com", "duckduckgo.com")
SEARCH_PATTERNS = ("/scholar?", "search?", "q=", "query=")

JOURNAL_HOSTS = (
    "thelancet.com", "nejm.org", "jamanetwork.com", "nature.com", "bmj.com",
    "sciencedirect.com", "onlinelibrary.wiley.com", "link.springer.com", "springer.com",
    "academic.oup.com", "annalsofoncology.org", "ejcancer.com", "aacrjournals.org",
    "ascopubs.org", "cochranelibrary.com", "ncbi.nlm.nih.gov", "pmc.ncbi.nlm.nih.gov",
    "esmo.org", "medscape.com", "cancer.net", "tripdatabase.com", "pubmed.ncbi.nlm.nih.gov",
)
JOURNAL_SEARCH_PATTERNS = ("/action/doSearch", "dosearch", "/search", "searchresults", "search?q=", "q=", "term=", "query=", "qs=")
ARTICLE_PATTERNS = ("/article/", "/articles/", "/fulltext", "/fullarticle", "/doi/", "/content/", "/science/article/", "/journals/", "/cdsr/", "/full", "/abstract", "/pdf/", "/fullarticle", "/article-abstract", "/pubmed/", "/pmc/articles/", "/pub/", "/view/")

COMPANY_HOSTS = (
    "pfizer.com", "merck.com", "novartis.com", "roche.com", "bms.com", "bristol-myers.com",
    "gsk.com", "glaxosmithkline.com", "sanofi.com", "astrazeneca.com", "abbvie.com",
    "amgen.com", "gilead.com", "biogen.com", "regeneron.com", "moderna.com",
    "biomarin.com", "vertex.com", "illumina.com", "qiagen.com", "janssen.com", "johnson-johnson.com",
)
COMPANY_DATA_PATTERNS = ("/disease/", "/indication/", "/therapeutic-area/", "/research/", "/data/", "/statistics/", "/epidemiology/", "/prevalence/", "/incidence/", "/burden/", "/patient/")

REFERENCE_PATTERNS = ("/reference", "/citation", "/cited-by", "/references", "/bibliography", "/doi/", "doi.org", "/cite", "/bib")

KPI_KEYWORDS = (
    "incidence", "prevalence", "cases", "patients", "new cases", "diagnosed",
    "per 100,000", "per 100000", "per 100k", "rate", "estimated", "annual",
    "mortality", "survival", "percent", "%", "percentage", "epidemiology",
    "burden", "prevalence rate", "incidence rate", "diagnosis", "morbidity"
)

NUMBER_PATTERN = re.compile(r"\b(\d{1,3}(?:[, \u00a0]\d{3})*(?:\.\d+)?|\d+(?:\.\d+)?)\b")
PERCENT_PATTERN = re.compile(r"\b(\d+(?:\.\d+)?)\s*%|\b(\d+(?:\.\d+)?)\s+percent\b")

In [ ]:
def fetch_page(url: str, retry: int = 0) -> Optional[str]:
    """Fetch page with retry logic - MATCHING WEB PLATFORM."""
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": USER_AGENT}, allow_redirects=True)
        r.raise_for_status()
        return r.text
    except Exception as e:
        if retry < 2:
            time.sleep(REQUEST_DELAY * (retry + 1))
            return fetch_page(url, retry + 1)
        return None

def is_plausible_datapoint(s: str, allow_small: bool = False) -> bool:
    """Check if number looks like epidemiology data - MATCHING WEB PLATFORM."""
    s_clean = s.replace("\u00a0", " ").replace(",", "").replace(" ", "").strip()
    try:
        n = float(s_clean)
        if 1900 <= n <= 2030 and "." not in s_clean:
            return False  # Year
        if n >= 100:
            return True  # Counts
        if 0.01 <= n <= 100:
            return True  # Rates/percentages
        if allow_small and 1 <= n < 100 and "." in s_clean:
            return True
        return False
    except ValueError:
        return False

def is_search_url(url: str) -> bool:
    """Check if URL is a search page."""
    try:
        parsed = urlparse(url)
        host = (parsed.netloc or "").lower().replace("www.", "")
        combined = (parsed.path or "").lower() + " " + (parsed.query or "").lower()
        return any(h in host for h in SEARCH_HOSTS) and any(p in combined for p in SEARCH_PATTERNS)
    except Exception:
        return False

def is_journal_search_url(url: str) -> bool:
    """Check if URL is a journal search page."""
    try:
        parsed = urlparse(url)
        host = (parsed.netloc or "").lower().replace("www.", "")
        combined = (parsed.path or "").lower() + " " + (parsed.query or "").lower()
        if not any(h in host for h in JOURNAL_HOSTS):
            return False
        search_lower = [p.lower() for p in JOURNAL_SEARCH_PATTERNS]
        return any(p in combined for p in search_lower)
    except Exception:
        return False

def is_company_site(url: str) -> bool:
    """Check if URL is a company/pharma site."""
    try:
        parsed = urlparse(url)
        host = (parsed.netloc or "").lower().replace("www.", "")
        return any(c in host for c in COMPANY_HOSTS)
    except Exception:
        return False

In [ ]:
def extract_from_tables(soup: BeautifulSoup) -> List[Tuple[str, str]]:
    """Extract datapoints from HTML tables - MATCHING WEB PLATFORM (35 tables)."""
    results = []
    for table in soup.find_all("table")[:35]:  # More tables like web platform
        for row in table.find_all("tr"):
            cells = row.find_all(["td", "th"])
            row_text = " ".join(c.get_text(separator=" ", strip=True) for c in cells).lower()
            for cell in cells:
                text = cell.get_text(separator=" ", strip=True)
                # Percentages
                for m in PERCENT_PATTERN.finditer(text):
                    g1, g2 = m.group(1), m.group(2)
                    v = (g1 or g2 or "").strip()
                    if v:
                        try:
                            if 0 <= float(v) <= 100:
                                results.append(("percent", f"{v}%"))
                        except ValueError:
                            pass
                # Numbers
                for m in NUMBER_PATTERN.finditer(text):
                    val = m.group(1)
                    if val and is_plausible_datapoint(val, allow_small=True):
                        label = "value"
                        if "incidence" in row_text and "prevalence" not in row_text:
                            label = "incidence"
                        elif "prevalence" in row_text:
                            label = "prevalence"
                        elif "cases" in row_text or "patients" in row_text:
                            label = "cases"
                        elif "rate" in row_text or "per 100" in row_text:
                            label = "rate"
                        elif "%" in row_text or "percent" in row_text:
                            label = "percent"
                        results.append((label, val))
    return results

def extract_percentages_from_text(text: str) -> List[Tuple[str, str]]:
    """Extract percentages from text."""
    results = []
    for m in PERCENT_PATTERN.finditer(text):
        g1, g2 = m.group(1), m.group(2)
        val = (g1 or g2 or "").strip()
        if val:
            try:
                n = float(val)
                if 0 <= n <= 100:
                    results.append(("percent", f"{val}%"))
            except ValueError:
                pass
    return results

def extract_numbers_from_text(text: str, require_kpi_context: bool = False) -> List[Tuple[str, str]]:
    """Extract numbers from text - MATCHING WEB PLATFORM (wider context)."""
    results = []
    text_lower = text.lower()
    for m in NUMBER_PATTERN.finditer(text):
        val = m.group(1)
        if not val or not is_plausible_datapoint(val, allow_small=True):
            continue
        if require_kpi_context:
            start = max(0, m.start() - 100)  # Wider context like web platform
            end = min(len(text), m.end() + 100)
            window = text_lower[start:end]
            if not any(kw in window for kw in KPI_KEYWORDS):
                continue
        label = "value"
        start = max(0, m.start() - 80)
        end = min(len(text), m.end() + 80)
        window = text_lower[start:end]
        if "incidence" in window and "prevalence" not in window:
            label = "incidence"
        elif "prevalence" in window:
            label = "prevalence"
        elif "cases" in window or "patients" in window or "new cases" in window:
            label = "cases"
        elif "rate" in window or "per 100" in window:
            label = "rate"
        elif "mortality" in window or "death" in window:
            label = "mortality"
        elif "survival" in window:
            label = "survival"
        results.append((label, val))
    return results

def extract_from_body_aggressive(soup: BeautifulSoup, body_text: str) -> List[Tuple[str, str]]:
    """Extract from body - MATCHING WEB PLATFORM (120k chars, 250 tags)."""
    found = []
    found.extend(extract_percentages_from_text(body_text))
    found.extend(extract_numbers_from_text(body_text[:120000], require_kpi_context=True))  # More text
    for tag in soup.find_all(["li", "dd", "p", "div", "span", "section"])[:250]:  # More tags
        t = tag.get_text(separator=" ", strip=True)
        if 10 <= len(t) <= 1500:
            found.extend(extract_percentages_from_text(t))
            found.extend(extract_numbers_from_text(t, require_kpi_context=False))
    return found

In [ ]:
def get_all_links_from_soup(soup: BeautifulSoup, base_url: str, max_links: int = 15, filter_patterns: Optional[List[str]] = None) -> List[str]:
    """Extract all relevant links - MATCHING WEB PLATFORM (200 links checked)."""
    links = []
    base_netloc = urlparse(base_url).netloc.lower().replace("www.", "")
    seen = set()
    
    for a in soup.find_all("a", href=True)[:200]:  # More links like web platform
        href = a.get("href", "").strip()
        if not href or href.startswith("#") or href.startswith("javascript:") or href.startswith("mailto:"):
            continue
        full = urljoin(base_url, href)
        if full in seen:
            continue
        try:
            p = urlparse(full)
            netloc = p.netloc.lower().replace("www.", "")
            path = p.path.lower()
            if base_netloc not in netloc and netloc not in base_netloc:
                continue
            if filter_patterns:
                if not any(pat in path or pat in href.lower() for pat in filter_patterns):
                    continue
            seen.add(full)
            links.append(full)
            if len(links) >= max_links:
                break
        except Exception:
            continue
    return links

def get_search_result_links(soup: BeautifulSoup, base_url: str) -> List[str]:
    """From Google Scholar/search results - MATCHING WEB PLATFORM."""
    links = []
    # Google Scholar: look for <h3><a>
    for h3 in soup.find_all("h3")[:20]:
        a = h3.find("a", href=True)
        if a:
            href = a.get("href", "")
            if href.startswith("http") or href.startswith("/"):
                full = urljoin(base_url, href)
                if full not in links:
                    links.append(full)
    # Also check <a> tags with text that looks like article titles
    for a in soup.find_all("a", href=True)[:80]:
        href = a.get("href", "")
        text = (a.get_text() or "").strip()
        if len(text) > 15 and len(text) < 250 and ("http" in href or href.startswith("/")):
            full = urljoin(base_url, href)
            if full not in links and not any(x in full.lower() for x in ["javascript:", "mailto:", "#"]):
                links.append(full)
    return links[:8]  # More results like web platform

def get_reference_links(soup: BeautifulSoup, base_url: str) -> List[str]:
    """From article page, find references - MATCHING WEB PLATFORM."""
    links = []
    # Look for sections with "reference", "citation", "bibliography"
    for tag in soup.find_all(["section", "div", "ul", "ol"], class_=re.compile(r"reference|citation|bibliography", re.I))[:5]:
        for a in tag.find_all("a", href=True)[:15]:
            href = a.get("href", "")
            if href.startswith("http") or href.startswith("/"):
                full = urljoin(base_url, href)
                if any(pat in full.lower() for pat in REFERENCE_PATTERNS) or "doi.org" in full.lower() or "pubmed" in full.lower():
                    if full not in links:
                        links.append(full)
    # Also check links with "doi" or "reference" in text
    for a in soup.find_all("a", href=True)[:120]:
        href = a.get("href", "")
        text = (a.get_text() or "").lower()
        if ("doi" in text or "reference" in text or "citation" in text or "pubmed" in text) and (href.startswith("http") or href.startswith("/")):
            full = urljoin(base_url, href)
            if full not in links:
                links.append(full)
    return links[:5]  # More references like web platform

In [ ]:
def fetch_and_extract(url: str) -> List[Tuple[str, str]]:
    """Fetch URL and extract datapoints (helper for immediate extraction)."""
    text = fetch_page(url)
    if not text:
        return []
    try:
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        body = soup.find("body") or soup
        body_text = body.get_text(separator=" ", strip=True) if body else ""
        found = []
        found.extend(extract_from_tables(soup))
        found.extend(extract_from_body_aggressive(soup, body_text))
        return found
    except Exception:
        return []

def parse_formatted_result(formatted_str: str) -> List[Tuple[str, str]]:
    """Parse formatted result string back into (label, value) tuples."""
    if not formatted_str:
        return []
    results = []
    # Format: "value1 (label1); value2 (label2); ..."
    parts = formatted_str.split(";")
    for part in parts:
        part = part.strip()
        if "(" in part and ")" in part:
            try:
                value = part.split("(")[0].strip()
                label = part.split("(")[1].split(")")[0].strip()
                if value and label:
                    results.append((label, value))
            except Exception:
                pass
    return results

def scrape_url(url: str, max_depth: int = 5, visited: Optional[Set[str]] = None, min_datapoints: int = 3) -> List[Tuple[str, str]]:
    """
    PERSISTENT multi-level extraction - EXACT MATCH TO WEB PLATFORM.
    Uses 6 strategies exactly as web platform:
    1. Extract from current page
    2. If search page → follow 8 result links + BOTH immediate AND recursive extraction
    3. If journal search → follow 8 article links + BOTH immediate AND recursive extraction
    4. If article page → follow 5 reference links
    5. If company site → scan 8 epidemiology pages
    6. Fallback: try any promising links with epidemiology keywords
    7. Final fallback: recursive extraction on promising links
    """
    if visited is None:
        visited = set()
    if url in visited or max_depth <= 0:
        return []
    visited.add(url)
    
    print(f"  → Scraping: {url[:70]}...")
    text = fetch_page(url)
    if not text:
        # Retry once (EXACT MATCH TO WEB PLATFORM)
        time.sleep(REQUEST_DELAY)
        text = fetch_page(url, retry=1)
        if not text:
            return []
    
    try:
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        body = soup.find("body") or soup
        body_text = body.get_text(separator=" ", strip=True) if body else ""
    except Exception as e:
        print(f"    ✗ Error parsing: {e}")
        return []
    
    # Extract from current page
    found = []
    found.extend(extract_from_tables(soup))
    found.extend(extract_from_body_aggressive(soup, body_text))
    
    if found:
        print(f"    ✓ Found {len(found)} datapoints on page")
    
    # Strategy 1: If search page, follow MORE result links (EXACT MATCH - 8 links, BOTH immediate + recursive)
    if is_search_url(url):
        print("    → Following search result links (8 links)...")
        result_links = get_search_result_links(soup, url)
        for link in result_links[:8]:
            if link not in visited and max_depth > 1:
                time.sleep(REQUEST_DELAY)
                # BOTH: immediate extraction AND recursive if needed (EXACT MATCH)
                sub_found = fetch_and_extract(link)
                found.extend(sub_found)
                # Also recursively follow if we still need more (EXACT MATCH)
                if len(found) < min_datapoints and max_depth > 2:
                    sub_result = scrape_url(link, max_depth - 1, visited, min_datapoints)
                    found.extend(sub_result)
                    if len(found) >= min_datapoints:
                        break
    
    # Strategy 2: If journal search page, follow MORE article links (EXACT MATCH - 8 links, BOTH immediate + recursive)
    if is_journal_search_url(url):
        print("    → Following article links (8 links)...")
        article_links = get_all_links_from_soup(soup, url, max_links=8, filter_patterns=list(ARTICLE_PATTERNS))
        for link in article_links:
            if link not in visited and max_depth > 1:
                time.sleep(REQUEST_DELAY)
                # BOTH: immediate extraction AND recursive if needed (EXACT MATCH)
                sub_found = fetch_and_extract(link)
                found.extend(sub_found)
                # Recursively follow references from articles (EXACT MATCH)
                if len(found) < min_datapoints and max_depth > 2:
                    sub_result = scrape_url(link, max_depth - 1, visited, min_datapoints)
                    found.extend(sub_result)
                    if len(found) >= min_datapoints:
                        break
    
    # Strategy 3: If article page, follow MORE references (EXACT MATCH - 5 links)
    if any(pat in url.lower() for pat in ARTICLE_PATTERNS):
        print("    → Following reference links (5 links)...")
        ref_links = get_reference_links(soup, url)
        for link in ref_links[:5]:
            if link not in visited and max_depth > 1:
                time.sleep(REQUEST_DELAY)
                sub_found = fetch_and_extract(link)
                found.extend(sub_found)
    
    # Strategy 4: If company site, scan MORE epidemiology pages (EXACT MATCH - 8 links)
    if is_company_site(url):
        print("    → Scanning company epidemiology pages (8 links)...")
        company_links = get_all_links_from_soup(soup, url, max_links=8, filter_patterns=list(COMPANY_DATA_PATTERNS))
        for link in company_links:
            if link not in visited and max_depth > 1:
                time.sleep(REQUEST_DELAY)
                sub_found = fetch_and_extract(link)
                found.extend(sub_found)
                if len(found) >= min_datapoints:
                    break
    
    # Strategy 5: Try ANY promising links with epidemiology keywords (EXACT MATCH - 10 links, fallback)
    if len(found) < min_datapoints:
        print("    → Trying fallback links with epidemiology keywords (10 links)...")
        all_links = get_all_links_from_soup(soup, url, max_links=10)
        for link in all_links:
            if link not in visited and max_depth > 1:
                link_lower = link.lower()
                if any(kw in link_lower for kw in ["epidemiology", "incidence", "prevalence", "statistics", "data", "burden", "patient", "disease"]):
                    time.sleep(REQUEST_DELAY)
                    sub_found = fetch_and_extract(link)
                    found.extend(sub_found)
                    if len(found) >= min_datapoints:
                        break
    
    # Strategy 6: If still not enough, try recursive extraction on promising links (EXACT MATCH)
    if len(found) < min_datapoints and max_depth > 2:
        print("    → Trying recursive extraction on promising links (3 links)...")
        promising = get_all_links_from_soup(soup, url, max_links=5)
        for link in promising[:3]:
            if link not in visited:
                time.sleep(REQUEST_DELAY * 1.5)  # Longer delay for recursive (EXACT MATCH)
                sub_result = scrape_url(link, max_depth - 2, visited, min_datapoints)
                found.extend(sub_result)
                if len(found) >= min_datapoints:
                    break
    
    return found

In [ ]:
# Main scraping loop - EXACT MATCH TO WEB PLATFORM'S deep_dive_link_record
all_results = []

for idx, source_url in enumerate(SOURCE_URLS, 1):
    print(f"\n{'='*80}")
    print(f"[{idx}/{len(SOURCE_URLS)}] Processing: {source_url[:70]}...")
    print(f"{'='*80}")
    
    # EXACT MATCH: Same logic as web platform's deep_dive_link_record
    # Step 1: Try with max_depth=5, min_datapoints=2
    print("  → Attempt 1: max_depth=5, min_datapoints=2")
    datapoints = scrape_url(source_url, max_depth=5, min_datapoints=2)
    
    # Step 2: If no result, retry with max_depth=4, min_datapoints=1 (EXACT MATCH)
    if not datapoints or len(datapoints) < 2:
        print("  → Attempt 2: max_depth=4, min_datapoints=1 (final attempt)")
        time.sleep(REQUEST_DELAY)
        datapoints = scrape_url(source_url, max_depth=4, min_datapoints=1)
    
    # Deduplicate (EXACT MATCH to web platform's _dedupe_and_format)
    seen = set()
    ordered = []
    # Order by priority exactly as web platform
    for priority_label in ("incidence", "prevalence", "cases", "rate", "percent", "mortality", "survival", "value"):
        for lbl, val in datapoints:
            if lbl == priority_label:
                vnorm = val.replace(",", "").replace(" ", "").replace("%", "")
                if vnorm not in seen:
                    seen.add(vnorm)
                    ordered.append((lbl, val))
    
    # Format result (EXACT MATCH: up to 30 if >15 found, else 25)
    max_vals = 30 if len(ordered) > 15 else 25
    ordered = ordered[:max_vals]
    
    if ordered:
        value_str = "; ".join([f"{v} ({l})" for l, v in ordered])
    else:
        value_str = "No datapoints found"
    
    all_results.append({
        "indication": INDICATION,
        "country": COUNTRY,
        "source_url": source_url,
        "value": value_str,
        "datapoint_count": len(ordered),
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
    
    print(f"\n✓ Extracted {len(ordered)} unique datapoints")
    if ordered:
        preview = value_str[:150] + "..." if len(value_str) > 150 else value_str
        print(f"  Preview: {preview}")
    else:
        print("  ⚠ No datapoints found after deep-dive")
    
    time.sleep(REQUEST_DELAY)  # Be polite between sources

In [ ]:
# Create DataFrame and save
df = pd.DataFrame(all_results)

print(f"\n{'='*80}")
print(f"SUMMARY")
print(f"{'='*80}")
print(f"Total sources processed: {len(df)}")
print(f"Total datapoints extracted: {df['datapoint_count'].sum()}")
print(f"Sources with datapoints: {(df['datapoint_count'] > 0).sum()}")

# Display results
display(df)

In [ ]:
# Save to CSV
output_file = f"epidemiology_scraped_{INDICATION.replace(' ', '_')}_{COUNTRY}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(output_file, index=False)
print(f"\n✓ Results saved to: {output_file}")

In [ ]:
# Optional: Detailed breakdown
print("\nDetailed breakdown:")
for idx, row in df.iterrows():
    print(f"\n{idx+1}. {row['source_url'][:60]}...")
    print(f"   Datapoints: {row['datapoint_count']}")
    if row['datapoint_count'] > 0:
        print(f"   Values: {row['value'][:150]}..." if len(row['value']) > 150 else f"   Values: {row['value']}")